# Data treatment

In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import csv

## Scrapt word ewe - englise data from `http://www.peterlin.pl/ewe/words.html`

In [2]:
word_url = "http://www.peterlin.pl/ewe/words.html"

res = requests.get(word_url)
res.encoding = "utf-8"
soup = BeautifulSoup(res.text)

data_raw = soup.find_all("dl", class_="letter")

# Liste qui contiendra les données
data = []

# Parcourir chaque bloc
for block in data_raw:

    ewe_words = block.find_all("dt")
    english_words = block.find_all("dd")

    # Associer chaque mot ewe à sa traduction
    for ewe, eng in zip(ewe_words, english_words):

        ewe_word = ewe.get_text(strip=True)
        english_word = eng.get_text(strip=True)

        data.append({
            "ewe": ewe_word,
            "english": english_word
        })

# Créer le DataFrame
df = pd.DataFrame(data)
df.to_csv("./peterlin/ewe_english_dictionary.csv", index=False, encoding="utf-8")
df.head(10)

,ewe,english
0,abati,bed
1,ablegɔ,chair
2,Ablotsi,Europe
3,ablɔ,street
4,ablɔɖe,freedom
5,abolo,bread
6,abɔ,arm
7,abɔta,shoulder
8,Abrã,name for a girl born on Tuesday
9,ade,six


## Paragraphes with thier translations from peterlin

In [3]:
urls = [
    "http://www.peterlin.pl/ewe/djoubogbe.html",
    "http://www.peterlin.pl/ewe/names-weekdays.html",
    "http://www.peterlin.pl/ewe/child-naming.html",
    "http://www.peterlin.pl/ewe/marriage.html",
    "http://www.peterlin.pl/ewe/funeral.html",
    "http://www.peterlin.pl/ewe/grandfather.html"
]

data = []

for url in urls:

    response = requests.get(url)
    response.encoding = "utf-8"

    soup = BeautifulSoup(response.text, "html.parser")

    sections = soup.find_all("h2")

    for h2 in sections:

        subject = h2.get_text(strip=True)

        ewe_text = ""
        english_text = ""

        current = h2.find_next_sibling()

        mode = None

        while current and current.name != "h2":

            # Détection des sous-sections
            if current.name == "h3":

                title = current.get_text(strip=True).lower()

                if "ewe version" in title:
                    mode = "ewe"

                elif "english translation" in title:
                    mode = "english"

            # Gestion des paragraphes texte ET poème
            elif current.name == "p":

                classes = current.get("class", [])

                if "text" in classes or "poem" in classes:

                    text = current.get_text(
                        separator="\n",
                        strip=True
                    )

                    if mode == "ewe":
                        ewe_text += text + "\n"

                    elif mode == "english":
                        english_text += text + "\n"

            current = current.find_next_sibling()

        # Nettoyage
        ewe_text = ewe_text.strip()
        english_text = english_text.strip()

        # Ignorer sections vides
        if ewe_text or english_text:

            data.append({
                "Subject": subject,
                "Ewe": ewe_text,
                "English": english_text,
                "Source_URL": url
            })

df = pd.DataFrame(data)

df.to_csv(
    "./peterlin/ewe_dataset.csv",
    index=False,
    encoding="utf-8-sig"
)

df.head()

,Subject,Ewe,English,Source_URL
0,Djoubogbe Kossi Afoutou's self-introduction,"Ŋkɔ nye enye AFOUTOU Djoubogbé Kossi, o dzim l...","My name is AFOUTOU Djoubogbé Kossi, I was born...",http://www.peterlin.pl/ewe/djoubogbe.html
1,Ewe personal names corresponding to various we...,KƆSIƉAME ŊKEKEWO KPLE EME NYAWO\nLe Eʋe dukɔwo...,DAYS IN THE WEEK WITH THEIR STEREOTIPES\nIn Ew...,http://www.peterlin.pl/ewe/names-weekdays.html
2,Story of naming a baby by a foreigner,Ne nyɔnu aɖe le evi dzim eye wo le kukum nɛ la...,If a woman often loss his baby after he is bor...,http://www.peterlin.pl/ewe/child-naming.html
3,Marriage in Ewe tradition,Esrɔ ɖeɖe le Eʋeawo ƒe dekɔnu wonu.\nLe dekɔnu...,Marriage according to Eʋe tradition.\nAccordin...,http://www.peterlin.pl/ewe/marriage.html
4,Funeral in Ewe tradition,ETSƆ WƆWƆ LE EƲE DUKƆWOME\r\n\r\nLe eʋe dukɔ w...,FUNERAL CEREMONIES IN EWE ETHNICAL GROUPS LAND...,http://www.peterlin.pl/ewe/funeral.html


In [4]:
url = "http://www.peterlin.pl/ewe/phrases.html"

r = requests.get(url)
r.encoding = r.apparent_encoding

soup = BeautifulSoup(r.text, "html.parser")

data = []

# Toutes les sections (Greetings, etc.)
sections = soup.find_all("dl", class_="letter")


for section in sections:

    # catégorie juste avant le bloc
    header = section.find_previous("h3", class_="letter-head")
    category = header.get_text(strip=True) if header else "Unknown"

    dts = section.find_all("dt")
    dds = section.find_all("dd")

    # sécurité si structure décalée
    for dt, dd in zip(dts, dds):

        ewe_phrase = dt.get_text(" ", strip=True)
        english = dd.get_text(" ", strip=True)

        data.append({
            "Category": category,
            "Ewe": ewe_phrase,
            "English": english
        })

df = pd.DataFrame(data)

df.to_csv("./peterlin/ewe_phrases.csv", index=False, encoding="utf-8-sig")

df.head(10)

,Category,Ewe,English
0,Greetings,Ŋdi,Good morning
1,Greetings,Ŋdɔ,Good day (used from around noon)
2,Greetings,Ŋdɔ na wo,Good day to you (na=to; wo=you)
3,Greetings,Fiɛyi,Good evening
4,Greetings,Aƒeame ɖe,how is your home (asking about family)
5,Greetings,Mefɔ nyui,I am doing well
6,Greetings,Ezã ne nyɔ,good night
7,Greetings,Eyi sɔ,See you tomorrow
8,Greetings,Mia dogo,Goodbye
9,Polite expressions,Aƒenɔ,"Mrs., Lady (a polite term of address)"


## Data from opus

In [5]:
with open('./opus/bible/bible-uedin.ee-en.ee', encoding='utf-8') as f:
    bible_ewe_lines = f.readlines()

print(len(bible_ewe_lines))

with open('./opus/bible/bible-uedin.ee-en.en', encoding='utf-8') as f:
    bible_english_lines = f.readlines()

print(len(bible_english_lines))

16001
16001


In [6]:
df_bible_ee_en = pd.DataFrame({
    "ewe": bible_ewe_lines,
    "english": bible_english_lines
})

df_bible_ee_en.to_csv("./opus/bible_ee_en.csv", index=False, encoding="utf-8-sig")
print(df_bible_ee_en.shape)
df_bible_ee_en.tail(100)

(16001, 2)


,ewe,english
15901,“Ne anyigbadzifia siwo wɔ ahasi kplii eye wokp...,"And the kings of the earth, who have committed..."
15902,Eƒe fuwɔame ado ŋɔdzi na wo ale gbegbe be woan...,"Standing afar off for the fear of her torment,..."
15903,“Anyigbadzisitsalawo afa avi eye woafa nɛ elab...,And the merchants of the earth shall weep and ...
15904,"Adzɔnu siawoe nye sika, klosalo, kpexɔasiwo, d...","The merchandise of gold, and silver, and preci..."
15905,"Bubuawoe nye atikeʋeʋĩ, sina-mɔn, detsiƒonuʋe...","And cinnamon, and odours, and ointments, and f..."
...,...,...
15996,"Gbɔgbɔ la kple ŋugbetɔ la gblɔ be, “Va!” Eye a...","And the Spirit and the bride say, Come. And le..."
15997,Mele nu xlɔ̃m ame sia ame si le agbalẽ sia me...,For I testify unto every man that heareth the ...
15998,Eye ne ame aɖe aɖe nya aɖewo le nyagblɔɖigbale...,And if any man shall take away from the words ...
15999,Ame si le ɖase ɖim le nu siawo ŋuti la gblɔ be...,"He which testifieth these things saith, Surely..."


In [7]:
with open('./opus/QED/QED.ee-en.ee', encoding='utf-8') as f:
    qed_ewe_lines = f.readlines()

print(len(qed_ewe_lines))

with open('./opus/QED/QED.ee-en.en', encoding='utf-8') as f:
    qed_english_lines = f.readlines()

print(len(qed_english_lines))

284
284


In [8]:
df_qed_ee_en = pd.DataFrame({
    "ewe": qed_ewe_lines,
    "english": qed_english_lines
})

df_qed_ee_en.to_csv("./opus/qed_ee_en.csv", index=False, encoding="utf-8-sig")
print(df_qed_ee_en.shape)
df_qed_ee_en.tail(100)

(284, 2)


,ewe,english
184,"Böylece,şişeleme tesisleri tüm dünyaya yayıldı.\n","Under his leadership, bottling plants began to..."
185,Ve Coca-Cola gerçek anlamda ilk dünya markası ...,And Coca-Cola became the first truly global br...
186,"Daha sonra bir 100 yıl içinde, formül hala giz...","Over a 100 years later, the formula is still a..."
187,Fakat;Coca-Cola popülerlik bir sır değil.\n,But the popularity of Coca-Cola is no secret.\n
188,Dünyanın en tanınmış markasıdır.\n,It's the most recognized trademark in the worl...
...,...,...
279,Sesi kısılmaya başladı için için\n,His voice was soft and very slow\n
280,Edgar Allen Poe'nun 'The raven:\n,As he quoted The Raven from Edgar Allen Poe:\n
281,"Kara Karga' sından bir... alıntısında.. "" Ruhu...","""And my soul from out that shadow That lies fl..."
282,Acaba yükselecek miydi?\n,Shall be lifted?\n


## Dataset from kagggle

In [13]:
kaggle_df = pd.read_csv("./kaggle/EWE_ENGLISH.csv")
kaggle_df = kaggle_df.drop('Unnamed: 0', axis=1)
print(kaggle_df.shape)
kaggle_df.head(10)

(28614, 2)


,EWE,ENGLISH
0,Ne nyɔnu aɖe le evi dzim eye wo le kukum nɛ la...,﻿If a woman often loss his baby after he is bo...
1,Ŋkɔ sia nye na ŋkɔ si ke ame bubu tsɔna na ɖev...,"This name comes from another person, which mea..."
2,Ame si hɔ ɖevi la ƒlela tsona ƒome bubu me alo...,This person must not be part of the whole fami...
3,Kɔnua wo yina ale: evinɔ si ga dzi ɖevi bubu a...,The ceremony is done as follow: the family of ...
4,Ne ame aɖe vayina to afimagodzi he kɔ ɖevia la...,When somebody passes through the road and find...
5,"Emegbe la ɖevila ƒe pomea ɖona to, he tɔna tek...","After then, the family simply waiting for info..."
6,Esiao kãtã vayina le ga ƒoƒo aɖeo me.\n,All this happen in some hours.
7,Ɖevi yeye la ƒe ƒomea nana mɔnu kpɔkpɔ amesi f...,The family gives an opportunity to the person ...
8,"Le ɣemayi mela ame si fɔ ɖevila, nana ŋkɔ bubu...",At this time this person gives a name he wants...
9,"Evi fɔla si trozu ɖevila ƒe tɔ evelia la, ateŋ...",This person who becomes the second father of t...


In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kuroio/ewe-english-train-csv")

print("Path to dataset files:", path)

/home/romaric/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2.26M/2.26M [00:02<00:00, 851kB/s]

Extracting files...
Path to dataset files: /home/romaric/.cache/kagglehub/datasets/kuroio/ewe-english-train-csv/versions/1


In [10]:
from datasets import load_dataset

# Load the datasetimport kagglehub

# Download latest version
path = kagglehub.dataset_download("kuroio/ewe-english-train-csv")

print("Path to dataset files:", path)
dataset = load_dataset("michsethowusu/ewe-sentiments-corpus")

# Access the data
print(dataset['train'][0])

# Check sentiment distribution
from collections import Counter
sentiments = [item['sentiment'] for item in dataset['train']]
print(Counter(sentiments))

/home/romaric/Documents/training/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'Ewe': 'Nya la do tso nye nu me le dzɔdzɔenyenye me,', 'sentiment': 'Negative'}
Counter({'Positive': 196712, 'Negative': 140776})
